# Bayesian statistics with high-redshift UV luminosity functions

### An astronomy short-course tutorial

This is a self-contained implementation of the UV luminosity-function (UVLF) statistics workflow. It reads the selected observational tables, posterior exports, and functions in `utilities/*.py`; it does **not** read or execute another notebook. The physical model, fitted redshifts, and top two joint models follow [Kar, Alam & Silk (2025), arXiv:2507.20606](https://arxiv.org/abs/2507.20606).

By the end, you should be able to:

1. write the Gaussian UVLF likelihood and connect it to $\chi^2$;
2. distinguish a best fit, a posterior distribution, and a posterior-predictive band;
3. use supplied posterior samples to make GetDist contour plots and understand when thinning is meaningful;
4. calculate and interpret AIC, AICc, BIC, and a DIC proxy for two physical models;
5. inspect both models against all 112 observations at $z=4,5,6,7,8,9,11,12.5,14,16$;
6. explain the Laplace approximation as “peak likelihood × posterior volume / prior volume”; and
7. explain why an ordinary `emcee` posterior chain is not, by itself, a direct numerical evidence calculation.


## Road map and scope

There are three connected examples.

- **Pedagogical warm-up:** a local $z=11$ five-parameter Eddington-bias chain demonstrates residuals and propagation of parameter uncertainty. It is not used for the joint-model evidence comparison.
- **Joint posterior analysis:** each 20,000-row randomized export contains one `loglike` value followed by the model parameters. We use these rows for summaries, GetDist contours, derived $\alpha(z)$ and $M_0(z)$ bands, a sampled maximum-likelihood check, and a covariance-Gaussian Laplace approximation.
- **Joint physical comparison:** Model 1 evolves only $\alpha(z)$ ($k=7$); Model 2 also evolves $M_0(z)$ ($k=9$). The tutorial evaluates both recorded best fits at every fitted redshift and compares their AIC/AICc/BIC penalties.

The complete selected data bundle is included under `tutorial_data/joint_uvlf/`. Its per-bin counts sum to $N=112$, the value used in the information criteria.

> **Which chains are needed?** AIC and BIC need only $\log \mathcal L_{\max}$, $k$, and $N$. Credible regions and GetDist need posterior samples. A standard posterior chain plus saved likelihoods can support diagnostics and approximations, but a defensible evidence calculation normally uses a dedicated method such as nested sampling or thermodynamic integration with explicit priors.


In [1]:
!pip install getdist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.0/836.0 kB 10.8 MB/s eta 0:00:00


In [2]:
!find /content/drive -iname "lf_mapping.py" 2>/dev/null

In [ ]:
!pip install hmf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.4 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
print(Path("/content/drive").exists())
print(list(Path("/content/drive").iterdir()) if Path("/content/drive").exists() else "not mounted")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import warnings
import sys

import numpy as np
import pandas as pd
from IPython import get_ipython

PROJECT = Path("/content/drive/MyDrive/Bayesian_Statistics_UVLF_Tutorial")

if not (PROJECT / "utilities" / "lf_mapping.py").exists():
    raise FileNotFoundError(f"Expected project contents not found at {PROJECT}")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Force a non-blocking inline backend in Jupyter and during nbconvert execution.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.signal import fftconvolve
from astropy.cosmology import FlatLambdaCDM

from utilities import hmf_cosmo
from utilities import star_formation as sf
from utilities import lf_mapping
from utilities import observation_data
from utilities import variability
from utilities import lf_processing

print(f"Project: {PROJECT}")

try:
    from getdist import MCSamples, plots
except ImportError as exc:
    raise ImportError("GetDist is required for the contour section: pip install getdist") from exc

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.labelsize": 11})
pd.set_option("display.max_columns", 30)

# Locate this project by its utility package and tutorial data, not by another notebook.
#candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
#PROJECT = next(
 #   (
  #      p for p in candidates
   #     if (p / "utilities" / "lf_mapping.py").exists()
    #    and (p / "tutorial_data" / "joint_uvlf").is_dir()
    #),
    #None,
#)
#if PROJECT is None:
 #   raise FileNotFoundError("Launch this notebook from the mcmc_parallel project directory.")

redshift_files = [f"tutorial_data/joint_uvlf/LF_z{z}.txt" for z in
                  ["4", "5", "6", "7", "8", "9", "11", "12.5", "14", "16"]]
required = [
    *redshift_files,
    "mcmc_samples_z11.txt", "z11_log_prob.txt",
    "uvlf_param_betaboundfree_epsilonfree_alpharedshift_m1sigmafree_HSTpowerlaw_emceeupdated_random_samples_Likelihood.txt",
    "uvlf_param_betaboundfree_epsilonfree_alphm1redshift_sigmafree_HSTpowerlaw_emceeupdated_random_samples.txt",
    "utilities/hmf_cosmo.py", "utilities/star_formation.py", "utilities/lf_mapping.py",
    "utilities/observation_data.py", "utilities/variability.py", "utilities/lf_processing.py",
]
missing = [name for name in required if not (PROJECT / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing tutorial inputs: {missing}")

print(f"Project: {PROJECT}")
print("All self-contained tutorial inputs and utility modules were found.")


## 1. Observations, residuals, and the likelihood

For observed number density $\phi_i$ and model prediction $\phi_{\rm model}(M_{{\rm UV},i},\theta)$, the analysis uses

$$\chi^2(\theta)=\sum_i \frac{[\phi_i-\phi_{\rm model}(M_{{\rm UV},i},\theta)]^2}{s_i^2}, \qquad \log\mathcal L(\theta)=-\frac12\chi^2(\theta)+C.$$

The selected files contain asymmetric upper and lower errors. Following the likelihood implementation, we make one effective Gaussian error,

$$s_i=\sqrt{(s_{i,+}^2+s_{i,-}^2)/2}.$$

This is a modelling decision. Strongly asymmetric detections, upper limits, shared cosmic-variance errors, or correlated measurements require a more appropriate likelihood or covariance matrix.


In [ ]:
JOINT_REDSHIFTS = (4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 11.0, 12.5, 14.0, 16.0)
EXPECTED_COUNTS = {4.0: 13, 5.0: 12, 6.0: 21, 7.0: 10, 8.0: 13,
                   9.0: 11, 11.0: 14, 12.5: 7, 14.0: 7, 16.0: 4}

def redshift_tag(redshift):
    return f"{float(redshift):g}"

def load_uvlf(path, redshift):
    'Load one selected joint-fit table and standardize asymmetric-error names.'
    values = np.loadtxt(path)
    if values.ndim != 2 or values.shape[1] != 4:
        raise ValueError(f"Expected four numeric columns in {path}")

    # The archived low-z and high-z tables use opposite error-column orderings.
    if float(redshift) <= 7:
        columns = ["Muv", "phi", "err_down", "err_up"]
    else:
        columns = ["Muv", "phi", "err_up", "err_down"]
    frame = pd.DataFrame(values, columns=columns)
    frame["redshift"] = float(redshift)
    frame["sigma_eff"] = np.sqrt((frame.err_up**2 + frame.err_down**2) / 2.0)
    if not np.all(np.isfinite(frame)) or np.any(frame[["phi", "err_up", "err_down"]] <= 0):
        raise ValueError(f"Non-finite or non-positive LF values in {path}")
    return frame.sort_values("Muv").reset_index(drop=True)

def log_safe_yerr(observations, lower_fraction=0.95):
    """Clip only the displayed lower bar so it stays positive on a log axis."""
    phi = observations.phi.to_numpy()
    lower = np.minimum(observations.err_down.to_numpy(), lower_fraction * phi)
    return np.vstack([lower, observations.err_up.to_numpy()])

joint_data_by_z = {
    z: load_uvlf(PROJECT / "tutorial_data" / "joint_uvlf" / f"LF_z{redshift_tag(z)}.txt", z)
    for z in JOINT_REDSHIFTS
}
observed_counts = {z: len(frame) for z, frame in joint_data_by_z.items()}
if observed_counts != EXPECTED_COUNTS:
    raise ValueError(f"Unexpected per-redshift counts: {observed_counts}")

joint_data = pd.concat(joint_data_by_z.values(), ignore_index=True)
N_JOINT = len(joint_data)
if N_JOINT != 112:
    raise ValueError(f"The joint likelihood requires N=112, found {N_JOINT}")

data_z11 = joint_data_by_z[11.0]
count_table = pd.DataFrame({"redshift": observed_counts.keys(), "N": observed_counts.values()})
display(count_table.T)
display(data_z11.head())
print(f"Total selected joint-fit measurements: N = {N_JOINT}")


## 2. A local Eddington-bias warm-up

The supplied $z=11$ teaching chain uses a phenomenological double-power-law UVLF,

$$\Phi(x)=\frac{\phi_\star}{10^{0.4(\alpha+1)(x-M_\star)}+10^{0.4(\beta+1)(x-M_\star)}}, \qquad x=-M_{\rm UV},$$

followed by convolution with a Gaussian of width $\sigma_{\rm UV}$. On a steep luminosity function, symmetric scatter moves more numerous faint galaxies into bright bins than the reverse: Eddington bias.

The five sampled parameters are $\theta=(\phi_\star,M_\star,\alpha,\beta,\sigma_{\rm UV})$. The DPL, convolution, interpolation, and $\chi^2$ calculation are implemented directly below; this section has no dependency on another notebook.


In [ ]:
def dpl_uvlf(x, phi_star, M_star, alpha, beta):
    faint = 10 ** (0.4 * (alpha + 1.0) * (x - M_star))
    bright = 10 ** (0.4 * (beta + 1.0) * (x - M_star))
    return phi_star / (faint + bright)

def convolved_dpl(x_grid, theta):
    phi_star, M_star, alpha, beta, sigma_uv = np.asarray(theta, dtype=float)
    unconvolved = dpl_uvlf(x_grid, phi_star, M_star, alpha, beta)
    offsets = x_grid - np.mean(x_grid)
    kernel = np.exp(-0.5 * (offsets / sigma_uv) ** 2)
    kernel /= kernel.sum()
    return fftconvolve(unconvolved, kernel, mode="same")

def model_at_observations(theta, observations, x_grid=None):
    if x_grid is None:
        x_grid = np.linspace(10.0, 28.0, 2200)
    curve = convolved_dpl(x_grid, theta)
    return np.interp(-observations.Muv.to_numpy(), x_grid, curve)

def chi2(theta, observations):
    prediction = model_at_observations(theta, observations)
    return np.sum(((observations.phi.to_numpy() - prediction) /
                   observations.sigma_eff.to_numpy()) ** 2)


In [ ]:
parameter_names = ["phi_star", "M_star", "alpha", "beta", "sigma_uv"]
chain = np.loadtxt(PROJECT / "mcmc_samples_z11.txt")
log_posterior = np.loadtxt(PROJECT / "z11_log_prob.txt")
valid = np.isfinite(log_posterior) & np.all(np.isfinite(chain), axis=1)
chain, log_posterior = chain[valid], log_posterior[valid]
if chain.shape[1] != len(parameter_names) or len(chain) != len(log_posterior):
    raise ValueError("The z=11 chain and log-posterior files are inconsistent.")

best_index = np.argmax(log_posterior)
best_fit = chain[best_index]
q16, q50, q84 = np.percentile(chain, [16, 50, 84], axis=0)
summary = pd.DataFrame({
    "parameter": parameter_names, "best_fit": best_fit, "median": q50,
    "minus_1sigma": q50 - q16, "plus_1sigma": q84 - q50,
})
display(summary)

chi2_min_z11 = chi2(best_fit, data_z11)
print(f"Samples retained: {len(chain):,}")
print(f"max log posterior = {log_posterior[best_index]:.4f}")
print(f"chi2 at the sampled best fit = {chi2_min_z11:.4f}")
print(f"Check -2 log posterior = {-2 * log_posterior[best_index]:.4f}")


In [ ]:
x_grid = np.linspace(10.0, 28.0, 2200)
best_curve = convolved_dpl(x_grid, best_fit)
best_at_data = model_at_observations(best_fit, data_z11, x_grid=x_grid)
residuals = (data_z11.phi.to_numpy() - best_at_data) / data_z11.sigma_eff.to_numpy()

fig, (ax, axr) = plt.subplots(2, 1, figsize=(7.2, 6.5), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(data_z11.Muv, data_z11.phi,
            yerr=log_safe_yerr(data_z11), fmt="o", capsize=3,
            color="black", label="observations")
ax.plot(-x_grid, best_curve, color="#0072B2", lw=2.2, label="sampled best fit")
ax.set(yscale="log", ylim=(3e-8, 3e-3))
ax.set_ylabel(r"$\Phi\;[\mathrm{Mpc}^{-3}\,\mathrm{mag}^{-1}]$")
ax.set_title(r"$z=11$: Eddington-convolved double power law")
ax.legend()

axr.axhline(0, color="0.3", lw=1)
axr.axhspan(-1, 1, color="0.8", alpha=0.35)
axr.plot(data_z11.Muv, residuals, "o", color="#D55E00")
axr.set_ylabel(r"residual / $s_i$")
axr.set_xlabel(r"$M_{\rm UV}$")
axr.set_xlim(-23.0, -17.5)
plt.tight_layout()
plt.show()


**What the figure says.** The Eddington-convolved DPL follows the $z=11$ measurements across the plotted magnitude range. The standardized residuals fluctuate around zero and remain within roughly $\pm1$ for the adopted effective errors, so this local example shows no obvious magnitude-dependent lack of fit. The brightest points have large uncertainties and therefore carry less statistical leverage than their visual prominence suggests.


## 3. The two joint posterior exports, GetDist, and thinning

A posterior chain approximates $p(\theta\mid D)$. It supports marginalized intervals, parameter correlations, derived quantities, and posterior-predictive calculations. It is **not required** for AIC or BIC if a reliable $\log\mathcal L_{\max}$ is already available.

Model 1's file contains `loglike` plus seven parameters. Model 2's contains `loglike` plus nine parameters, adding $M_2$ and $M_3$ for $\log M_0(z)=M_1+M_2z+M_3z^2$. These are randomized/post-processed exports with no step, walker, temperature, nested-sampling weight, or prior-sample columns. They support posterior plots, sampled likelihood maxima, a DIC-style proxy, and approximation methods, but they cannot reconstruct trace plots, autocorrelation time, $\hat R$, thermodynamic integration, or a direct evidence estimate.

Further sequential thinning has no statistical benefit here. `plot_stride` below is only optional plot decimation. Convergence must be assessed on the original unflattened chains before creating a randomized export.


In [ ]:
def load_random_likelihood_export(path, expected_parameters):
    table = np.genfromtxt(path, names=True, delimiter="\t")
    columns = list(table.dtype.names)
    expected_columns = ["loglike"] + list(expected_parameters)
    if columns != expected_columns:
        raise ValueError(f"Unexpected columns in {Path(path).name}: {columns}")
    loglike = np.asarray(table["loglike"], dtype=float)
    samples = np.column_stack([table[name] for name in expected_parameters])
    valid = np.isfinite(loglike) & np.all(np.isfinite(samples), axis=1)
    return samples[valid], loglike[valid]

joint_names = ["m1", "alpha0", "alpha1", "alpha2", "beta0", "sigma0", "epsilon0"]
model2_names = ["m1", "m2", "m3", "alpha0", "alpha1", "alpha2", "beta0", "sigma0", "epsilon0"]
joint_chain_path = PROJECT / "uvlf_param_betaboundfree_epsilonfree_alpharedshift_m1sigmafree_HSTpowerlaw_emceeupdated_random_samples_Likelihood.txt"
model2_chain_path = PROJECT / "uvlf_param_betaboundfree_epsilonfree_alphm1redshift_sigmafree_HSTpowerlaw_emceeupdated_random_samples.txt"
joint_chain, joint_loglike = load_random_likelihood_export(joint_chain_path, joint_names)
model2_chain, model2_loglike = load_random_likelihood_export(model2_chain_path, model2_names)

def summarize_likelihood_export(samples, loglike, names, model_label):
    best_index = np.argmax(loglike)
    q16, q50, q84 = np.percentile(samples, [16, 50, 84], axis=0)
    table = pd.DataFrame({
        "parameter": names, "best_sample": samples[best_index], "median": q50,
        "minus_1sigma": q50 - q16, "plus_1sigma": q84 - q50,
    })
    print(f"{model_label}: {len(samples):,} rows, sampled max loglike={loglike[best_index]:.6f}, "
          f"sampled chi2={-2 * loglike[best_index]:.6f}")
    display(table)
    return best_index

joint_best_index = summarize_likelihood_export(joint_chain, joint_loglike, joint_names, "Model 1")
model2_best_index = summarize_likelihood_export(model2_chain, model2_loglike, model2_names, "Model 2")
joint_sampled_loglike_max = joint_loglike[joint_best_index]
model2_sampled_loglike_max = model2_loglike[model2_best_index]

# Both files are already randomized. This is plot decimation, not MCMC thinning.
plot_stride = 2
gd_model1 = MCSamples(
    samples=joint_chain[::plot_stride], loglikes=-joint_loglike[::plot_stride],
    names=joint_names,
    labels=[r"M_1", r"\alpha_0", r"\alpha_1", r"\alpha_2", r"\beta_0", r"\sigma_0", r"\epsilon_0"],
    label=r"Model 1: $\alpha(z)$", settings={"ignore_rows": 0},
)
gd_model2 = MCSamples(
    samples=model2_chain[::plot_stride], loglikes=-model2_loglike[::plot_stride],
    names=model2_names,
    labels=[r"M_1", r"M_2", r"M_3", r"\alpha_0", r"\alpha_1", r"\alpha_2", r"\beta_0", r"\sigma_0", r"\epsilon_0"],
    label=r"Model 2: $\alpha(z)+M_0(z)$", settings={"ignore_rows": 0},
)
print(f"GetDist uses {len(joint_chain[::plot_stride]):,} Model-1 rows and "
      f"{len(model2_chain[::plot_stride]):,} Model-2 rows (plot_stride={plot_stride}).")


In [ ]:
common_parameters = ["alpha0", "alpha1", "alpha2", "beta0", "sigma0", "epsilon0"]
g = plots.get_subplot_plotter(width_inch=11)
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add = 0.55
g.triangle_plot(
    [gd_model1, gd_model2], params=common_parameters, filled=True,
    contour_colors=["#0072B2", "#D55E00"], title_limit=1,
)
plt.show()


**What the common-parameter contours say.** The 68% and 95% credible regions for the shared parameters largely overlap, so allowing $M_0(z)$ to evolve does not substantially displace their marginalized constraints. Long tilted contours among $\alpha_0$, $\alpha_1$, and $\alpha_2$ show strong polynomial-coefficient covariance. Trade-offs involving $\sigma_0$ and $\epsilon_0$ illustrate how UV scatter and intrinsic efficiency can compensate one another. Model 2 is generally broader because its extra mass-evolution freedom weakens some shared-parameter constraints.


In [ ]:
g_mass = plots.get_subplot_plotter(width_inch=6)
g_mass.triangle_plot(
    [gd_model2], params=["m1", "m2", "m3"], filled=True,
    contour_colors=["#D55E00"], title_limit=1,
)
plt.show()


**What the Model-2 mass contours say.** $M_1$, $M_2$, and $M_3$ are strongly correlated and their posteriors are curved/non-Gaussian. Individual coefficients are therefore not independently well determined: different triples produce similar $M_0(z)$ over the redshift range constrained by the data. Interpret the derived $M_0(z)$ band below rather than any coefficient alone. This geometry is also a warning that a covariance-based Laplace evidence can be fragile.


In [ ]:
z_posterior = np.linspace(4, 16, 160)

def derived_evolution(samples, names, z):
    index = {name: i for i, name in enumerate(names)}
    alpha = (samples[:, index["alpha0"], None]
             + samples[:, index["alpha1"], None] * z[None, :]
             + samples[:, index["alpha2"], None] * z[None, :]**2)
    if "m2" in index:
        logM0 = (samples[:, index["m1"], None]
                 + samples[:, index["m2"], None] * z[None, :]
                 + samples[:, index["m3"], None] * z[None, :]**2)
    else:
        logM0 = np.repeat(samples[:, index["m1"], None], len(z), axis=1)
    return (np.percentile(alpha, [16, 50, 84], axis=0),
            np.percentile(logM0, [16, 50, 84], axis=0))

alpha_model1, logM0_model1 = derived_evolution(joint_chain, joint_names, z_posterior)
alpha_model2, logM0_model2 = derived_evolution(model2_chain, model2_names, z_posterior)
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2))
for label, alpha_q, mass_q, color in [
    ("Model 1", alpha_model1, logM0_model1, "#0072B2"),
    ("Model 2", alpha_model2, logM0_model2, "#D55E00"),
]:
    axes[0].fill_between(z_posterior, alpha_q[0], alpha_q[2], color=color, alpha=0.22)
    axes[0].plot(z_posterior, alpha_q[1], color=color, lw=2, label=label)
    axes[1].fill_between(z_posterior, mass_q[0], mass_q[2], color=color, alpha=0.22)
    axes[1].plot(z_posterior, mass_q[1], color=color, lw=2, label=label)
axes[0].set(xlabel="redshift", ylabel=r"$\alpha(z)$", title="Low-mass SFE slope")
axes[1].set(xlabel="redshift", ylabel=r"$\log_{10}(M_0/M_\odot)$", title="Characteristic halo mass")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


**What the derived-evolution plot says.** Both models infer a decreasing numerical value of $\alpha(z)$ over $4\leq z\leq16$ and agree within their 68% bands. Model 1 fixes $\log_{10}M_0\simeq11.3$, whereas Model 2 permits an evolving median with rapidly widening uncertainty at high redshift; the constant-$M_0$ trajectory remains broadly compatible with that band. The derived curves are much easier to interpret than the highly covariant polynomial coefficients.


### Posterior uncertainty propagated to the warm-up UVLF

The next band propagates parameter uncertainty through the convolved DPL. It is a credible band for the latent mean UVLF, not a full posterior-predictive band: a full predictive distribution would also draw measurement noise and any intrinsic sampling/count model.


In [ ]:
rng = np.random.default_rng(25072025)
draw_indices = rng.choice(len(chain), size=250, replace=False)
curve_draws = np.array([convolved_dpl(x_grid, theta) for theta in chain[draw_indices]])
band16, band50, band84 = np.percentile(curve_draws, [16, 50, 84], axis=0)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.fill_between(-x_grid, band16, band84, color="#56B4E9", alpha=0.35,
                label="68% parameter band")
ax.plot(-x_grid, band50, color="#0072B2", lw=2, label="posterior median curve")
ax.errorbar(data_z11.Muv, data_z11.phi,
            yerr=log_safe_yerr(data_z11),
            fmt="o", capsize=3, color="black", label="observations")
ax.set(yscale="log", xlim=(-23.0, -17.5), ylim=(3e-8, 3e-3), xlabel=r"$M_{\rm UV}$",
       ylabel=r"$\Phi\;[\mathrm{Mpc}^{-3}\,\mathrm{mag}^{-1}]$")
ax.legend()
plt.tight_layout()
plt.show()


**What the uncertainty band says.** The median UVLF follows the $z=11$ data, and the parameter-only 68% band is tightest where the measurements are most informative. It widens toward the sparsely constrained bright end. This is uncertainty in the latent mean curve, not a posterior-predictive interval; measurement or count noise would make a predictive band wider.


## 4. The physical UVLF model and all joint redshifts

The paper maps halos to UV luminosity through

$$\epsilon(M_{\rm H})=\frac{2\epsilon_0}{(M_{\rm H}/M_0)^{-\alpha}+(M_{\rm H}/M_0)^\beta},\quad
\mathrm{SFR}=\epsilon f_b\dot M_{\rm H},\quad
M_{\rm H}\rightarrow M_{\rm UV}\rightarrow\Phi(M_{\rm UV}).$$

The local modules divide responsibility as follows:

| Module | Role in this notebook |
|---|---|
| `hmf_cosmo.py` | halo mass function and cosmology conversion |
| `star_formation.py` | accretion, double-power-law SFE, SFR-to-UV conversion, and dust |
| `lf_mapping.py` | $M_{\rm H}\to M_{\rm UV}$ map and numerical Jacobian |
| `observation_data.py` | loaders for broader observational compilations; the exact selected 112 points are loaded directly instead |
| `variability.py` | optional halo-mass-dependent scatter prescriptions |
| `lf_processing.py` | additional local processing utilities |

The joint chain used a normalized Gaussian FFT convolution on the mapped magnitude grid. That short operation is implemented explicitly below, alongside the utility calls, so no other notebook is needed. A bundled deterministic HMF cache was generated with the checked-in utility at the analysis mass resolution; if it is absent, the same cell recomputes all ten grids.

> **Reproduction audit.** The bundled files reproduce the exact ten-bin, $N=112$ selection. The current `hmf_cosmo.py` hard-codes the Behroozi HMF calibration, while the paper specifies Tinker. The standalone likelihood file in this checkout also contains stale configuration/signature choices. Consequently, the curves below are a close local reconstruction of the recorded best fits, not a bit-for-bit publication pipeline. The per-bin and total $\chi^2$ checks quantify the remaining difference instead of hiding it.


In [ ]:
# Best-fit values recorded in the paper's Table 3 / local AIC analysis.
MODEL1_BEST = dict(
    logM0=11.305802, alpha0=0.909616, alpha1=-0.019125, alpha2=-0.001544,
    beta=0.479666, sigma_uv=0.318659, epsilon0=0.271792,
)
MODEL2_BEST = dict(
    M1=10.803332, M2=0.087894, M3=0.004449,
    alpha0=1.183030, alpha1=-0.091760, alpha2=0.001796,
    beta=0.451845, sigma_uv=0.438679, epsilon0=0.263616,
)
RECORDED_JOINT_CHI2 = {"Model 1": 146.244894, "Model 2": 141.447111}

def physical_parameters_at_z(parameters, redshift):
    alpha = (parameters["alpha0"] + parameters["alpha1"] * redshift
             + parameters["alpha2"] * redshift**2)
    if "logM0" in parameters:
        logM0 = parameters["logM0"]
    else:
        logM0 = (parameters["M1"] + parameters["M2"] * redshift
                 + parameters["M3"] * redshift**2)
    return alpha, logM0

z_grid = np.linspace(4, 16, 200)
mass_grid = np.logspace(8, 13, 300)
fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8))
for label, parameters, color in [
    (r"Model 1: $\alpha(z)$", MODEL1_BEST, "#0072B2"),
    (r"Model 2: $\alpha(z),M_0(z)$", MODEL2_BEST, "#D55E00"),
]:
    states = np.array([physical_parameters_at_z(parameters, z) for z in z_grid])
    axes[0].plot(z_grid, states[:, 0], color=color, lw=2, label=label)
    axes[1].plot(z_grid, states[:, 1], color=color, lw=2, label=label)
    alpha11, logM011 = physical_parameters_at_z(parameters, 11.0)
    sfe11 = sf.star_formation_efficiency_fiducial(
        mass_grid, parameters["epsilon0"], 10**logM011, alpha11, parameters["beta"]
    )
    axes[2].plot(mass_grid, sfe11, color=color, lw=2, label=label)

axes[0].set(xlabel="redshift", ylabel=r"$\alpha(z)$", title="Low-mass slope")
axes[1].set(xlabel="redshift", ylabel=r"$\log_{10}(M_0/M_\odot)$", title="Mass scale")
axes[2].set(xscale="log", xlabel=r"$M_{\rm H}/M_\odot$",
            ylabel=r"$\epsilon(M_{\rm H},z=11)$", title="$z=11$ SFE")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()


**What the physical-parameter plot says.** These are recorded best-fit trajectories, not posterior medians. Both fits make the numerical value of $\alpha(z)$ decrease. Model 2 uses its extra freedom to move the characteristic halo mass strongly upward with redshift. At $z=11$ this shifts the SFE peak to a larger halo mass while leaving a similar peak efficiency. Because the posterior contours above are broad and non-Gaussian, the orange best-fit trajectory should not be mistaken for the typical inferred history.


In [ ]:
parameters_std = {
    "hubble": 0.6781, "Om0": 0.309, "Omcdm": 0.1191, "Omb": 2.249 / 100.0,
    "As": 2.092e-9, "ns": 0.9747, "S8": 0.821, "tau": 0.051,
}
parameters_std["fbaryon"] = parameters_std["Omb"] / (
    parameters_std["Omb"] + parameters_std["Omcdm"]
)
parameters_std["sigma8"] = parameters_std["S8"] / (parameters_std["Om0"] / 0.3) ** 0.5
cosmo_std = FlatLambdaCDM(
    H0=100 * parameters_std["hubble"], Om0=parameters_std["Om0"],
    Ob0=parameters_std["Omb"] / parameters_std["hubble"]**2,
    Tcmb0=2.7255, Neff=3.046, m_nu=[0.0, 0.0, 0.06],
)

def gaussian_fft_convolution(muv, phi_no_scatter, sigma_uv):
    'Normalized Gaussian FFT convolution used by the joint-chain analysis path.'
    offsets = np.asarray(muv) - np.mean(muv)
    kernel = np.exp(-0.5 * (offsets / sigma_uv) ** 2)
    kernel /= kernel.sum()
    return fftconvolve(phi_no_scatter, kernel, mode="same")

HMF_MASS_RESOLUTION = 0.002
HMF_CACHE_PATH = PROJECT / "tutorial_data" / "joint_uvlf" / "hmf_behroozi_planck_dlog0p002.npz"
joint_hmf = {}
if HMF_CACHE_PATH.exists():
    with np.load(HMF_CACHE_PATH) as cached_hmf:
        if not np.isclose(float(cached_hmf["mass_resolution"]), HMF_MASS_RESOLUTION):
            raise ValueError("The cached HMF mass resolution is inconsistent.")
        for z in JOINT_REDSHIFTS:
            tag = redshift_tag(z).replace(".", "p")
            joint_hmf[z] = (cached_hmf[f"log_mhalo_z{tag}"],
                            cached_hmf[f"phi_halo_z{tag}"],
                            cached_hmf[f"dndm_z{tag}"])
    print(f"Loaded deterministic HMF cache: {HMF_CACHE_PATH.name}")
else:
    print("HMF cache not found; calculating all ten grids with utilities.hmf_cosmo.")
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="'extrapolate_with_eh' was not set.*")
        for z in JOINT_REDSHIFTS:
            joint_hmf[z] = hmf_cosmo.calculate_hmf_kmaxfixed(
                z=z, Mmin=8, Mmax=13, dlog10m=HMF_MASS_RESOLUTION,
                cosmo=cosmo_std, parameters=parameters_std,
            )

def physical_uvlf_curve(redshift, parameters, hmf_entry=None):
    if hmf_entry is None:
        hmf_entry = joint_hmf[float(redshift)]
    log_mhalo, phi_halo, _dndm = hmf_entry
    alpha, logM0 = physical_parameters_at_z(parameters, redshift)
    muv = lf_mapping.mapfunc_mhalo_to_muv(
        log_mhalo, parameters_std, cosmo_std, "this_work", parameters["epsilon0"],
        10**logM0, alpha, parameters["beta"], redshift, include_dust=True,
    )
    jacobian = lf_mapping.mapfunc_jacobian_numeric(
        log_mhalo, parameters_std, cosmo_std, "this_work", parameters["epsilon0"],
        10**logM0, alpha, parameters["beta"], redshift, include_dust=True,
    )
    phi_no_scatter = phi_halo / jacobian
    phi_scattered = gaussian_fft_convolution(muv, phi_no_scatter, parameters["sigma_uv"])
    return muv, phi_scattered

def interpolate_curve_at_data(muv, phi_model, observations):
    order = np.argsort(muv)
    return np.interp(observations.Muv.to_numpy(), muv[order], phi_model[order])

model_specs = [
    ("Model 1", MODEL1_BEST, "#0072B2"),
    ("Model 2", MODEL2_BEST, "#D55E00"),
]
physical_curves = {name: {} for name, _, _ in model_specs}
chi2_by_redshift = []

fig, axes = plt.subplots(2, 5, figsize=(18, 7.2), sharey=True)
for ax, z in zip(axes.flat, JOINT_REDSHIFTS):
    observations = joint_data_by_z[z]
    ax.errorbar(
        observations.Muv, observations.phi,
        yerr=log_safe_yerr(observations),
        fmt="o", ms=3.8, capsize=2, color="black", label="selected data", zorder=3,
    )
    row = {"redshift": z, "N": len(observations)}
    for model_name, parameters, color in model_specs:
        muv, phi_model = physical_uvlf_curve(z, parameters)
        physical_curves[model_name][z] = (muv, phi_model)
        order = np.argsort(muv)
        ax.plot(muv[order], phi_model[order], color=color, lw=1.8, label=model_name)
        prediction = interpolate_curve_at_data(muv, phi_model, observations)
        row[f"chi2_{model_name.replace(' ', '_').lower()}"] = np.sum(
            ((observations.phi.to_numpy() - prediction) /
             observations.sigma_eff.to_numpy()) ** 2
        )
    chi2_by_redshift.append(row)
    ax.set(yscale="log", ylim=(3e-9, 4e-1))
    ax.set_xlim(observations.Muv.min() - 0.6, observations.Muv.max() + 0.6)
    ax.set_title(f"z = {redshift_tag(z)}  (N={len(observations)})")
    ax.set_xlabel(r"$M_{\rm UV}$")
    ax.tick_params(axis="both", labelsize=8)
axes[0, 0].set_ylabel(r"$\Phi\;[\mathrm{Mpc}^{-3}\,\mathrm{mag}^{-1}]$")
axes[1, 0].set_ylabel(r"$\Phi\;[\mathrm{Mpc}^{-3}\,\mathrm{mag}^{-1}]$")
axes[0, 0].legend(fontsize=7, loc="lower right")
fig.suptitle("Both recorded best fits across every redshift in the joint N=112 likelihood", y=1.02)
plt.tight_layout()
plt.show()

chi2_by_redshift = pd.DataFrame(chi2_by_redshift)
chi2_columns = ["chi2_model_1", "chi2_model_2"]
local_totals = chi2_by_redshift[chi2_columns].sum()
chi2_check = pd.DataFrame({
    "model": ["Model 1", "Model 2"],
    "local_reconstruction_chi2": local_totals.to_numpy(),
    "recorded_full_chain_chi2": [RECORDED_JOINT_CHI2["Model 1"], RECORDED_JOINT_CHI2["Model 2"]],
})
chi2_check["difference"] = (
    chi2_check.local_reconstruction_chi2 - chi2_check.recorded_full_chain_chi2
)
display(chi2_by_redshift.round(3))
display(chi2_check.round(3))

# Demonstrate the optional variability module without changing the constant-scatter fits above.
sigma_g24_z11 = variability.sigma_uv_vs_mhalo_G24(joint_hmf[11.0][0])
print(f"Optional Gelli+24 mass-dependent scatter spans {sigma_g24_z11.min():.3f}–"
      f"{sigma_g24_z11.max():.3f} mag on the z=11 halo grid (not used in these fits).")


**What the ten fit panels say.** Every redshift contributing to $N=112$ is now shown. Both models track the large decline in number density toward brighter magnitudes and higher redshift. Model 2 improves the reconstructed total $\chi^2$ but not every individual bin; its extra $M_0(z)$ freedom is therefore a global trade-off rather than a uniform improvement. The highest-redshift bins contain few, uncertain points, so their curves are much less tightly anchored than the low-redshift fits.

The local totals are close to, but not identical to, the recorded full-chain values. That small residual is the visible reproduction warning caused by the checked-in HMF/code-version mismatch described above. The AIC/BIC section therefore uses the recorded full-chain likelihood maxima, not these diagnostic totals.


## 5. Model complexity versus fit quality: AIC, AICc, BIC, and a DIC proxy

For $k$ fitted parameters and $N$ observations,

$$\mathrm{AIC}=-2\log\mathcal L_{\max}+2k,$$

$$\mathrm{AICc}=\mathrm{AIC}+\frac{2k(k+1)}{N-k-1},$$

$$\mathrm{BIC}=-2\log\mathcal L_{\max}+k\log N.$$

Smaller is preferred, but only differences between models fitted to the **same data with the same likelihood** are meaningful. AIC targets expected predictive performance; BIC uses a stronger sample-size-dependent penalty and has a large-sample connection to evidence. Neither is an absolute goodness-of-fit test.

The paper defines DIC using the deviance at the posterior mean, $D(\bar\theta)$. The saved likelihood exports do not provide an executable likelihood value exactly at $\bar\theta$, so the code below reproduces the repository's *best-deviance proxy*: $p_D=\bar D-D_{\rm best}$ and $\mathrm{DIC}_{\rm best}=D_{\rm best}+2p_D$. It is labelled as a proxy and should not be confused with an independently recomputed publication DIC.

The headline $\chi^2$ values are the best likelihoods recorded from the full-chain analysis in `AIC.ipynb`. A maximum from a 20,000-row randomized subset is expected to be slightly worse because the exact best row may not have been retained.


In [ ]:
def information_criteria(loglike_max, k, n):
    minus2loglike = -2.0 * loglike_max
    aic = minus2loglike + 2.0 * k
    aicc = aic + 2.0 * k * (k + 1.0) / (n - k - 1.0)
    bic = minus2loglike + k * np.log(n)
    return aic, aicc, bic

comparison = pd.DataFrame({
    "model": [r"Model 1: $\alpha(z)$", r"Model 2: $\alpha(z)+M_0(z)$"],
    "k": [7, 9], "N": [N_JOINT, N_JOINT],
    "chi2_min": [RECORDED_JOINT_CHI2["Model 1"], RECORDED_JOINT_CHI2["Model 2"]],
})
comparison["loglike_max"] = -0.5 * comparison.chi2_min
criteria = [information_criteria(row.loglike_max, row.k, row.N)
            for row in comparison.itertuples()]
comparison[["AIC", "AICc", "BIC"]] = np.asarray(criteria)
comparison["naive_dof"] = comparison.N - comparison.k
comparison["chi2_per_naive_dof"] = comparison.chi2_min / comparison.naive_dof
for criterion in ["AIC", "AICc", "BIC"]:
    comparison[f"delta_{criterion}"] = comparison[criterion] - comparison[criterion].min()
    raw_weight = np.exp(-0.5 * comparison[f"delta_{criterion}"])
    comparison[f"weight_{criterion}"] = raw_weight / raw_weight.sum()

columns = ["model", "k", "N", "chi2_min", "loglike_max", "AIC", "delta_AIC",
           "AICc", "delta_AICc", "BIC", "delta_BIC", "weight_AIC", "weight_BIC"]
display(comparison[columns].round(3))

sample_aic1, sample_aicc1, sample_bic1 = information_criteria(
    joint_sampled_loglike_max, k=7, n=N_JOINT
)
sample_aic2, sample_aicc2, sample_bic2 = information_criteria(
    model2_sampled_loglike_max, k=9, n=N_JOINT
)
sample_based_comparison = pd.DataFrame([
    ["Model 1", "20,000-row random export", joint_sampled_loglike_max,
     -2*joint_sampled_loglike_max, sample_aic1, sample_bic1],
    ["Model 1", "full-chain recorded maximum", comparison.loc[0, "loglike_max"],
     comparison.loc[0, "chi2_min"], comparison.loc[0, "AIC"], comparison.loc[0, "BIC"]],
    ["Model 2", "20,000-row random export", model2_sampled_loglike_max,
     -2*model2_sampled_loglike_max, sample_aic2, sample_bic2],
    ["Model 2", "full-chain recorded maximum", comparison.loc[1, "loglike_max"],
     comparison.loc[1, "chi2_min"], comparison.loc[1, "AIC"], comparison.loc[1, "BIC"]],
], columns=["model", "source", "loglike_max", "chi2_min", "AIC", "BIC"])
display(sample_based_comparison.round(4))

def best_deviance_dic_proxy(loglike):
    deviance = -2.0 * np.asarray(loglike)
    D_best, D_bar = deviance.min(), deviance.mean()
    p_D = D_bar - D_best
    return D_best + 2.0*p_D, D_best, D_bar, p_D

proxy1, D_best1, D_bar1, p_D1 = best_deviance_dic_proxy(joint_loglike)
proxy2, D_best2, D_bar2, p_D2 = best_deviance_dic_proxy(model2_loglike)
dic_proxy_comparison = pd.DataFrame({
    "model": ["Model 1", "Model 2"], "D_best": [D_best1, D_best2],
    "mean_D": [D_bar1, D_bar2], "p_D_best_proxy": [p_D1, p_D2],
    "DIC_best_proxy": [proxy1, proxy2],
})
display(dic_proxy_comparison.round(3))
print("Use the full-chain recorded maxima for the headline AIC/BIC table; a random export rarely contains the exact best row.")
print("The displayed DIC quantity is the repository's best-deviance proxy, not D(theta_bar)-based DIC.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
short_names = [r"$\alpha(z)$", r"$\alpha(z)+M_0(z)$"]
colors = ["#0072B2", "#D55E00"]
axes[0].bar(short_names, comparison.chi2_min, color=colors)
axes[0].set_ylabel(r"$\chi^2_{\min}=-2\log\mathcal{L}_{\max}$")
axes[0].set_title("Fit term: lower is better")
axes[0].set_ylim(138, 149)

x = np.arange(2)
width = 0.34
axes[1].bar(x - width/2, comparison.delta_AIC, width,
            label=r"$\Delta$AIC", color="#56B4E9")
axes[1].bar(x + width/2, comparison.delta_BIC, width,
            label=r"$\Delta$BIC", color="#E69F00")
axes[1].axhline(5, color="0.35", ls="--", lw=1, label=r"$\Delta=5$ guide")
axes[1].set(xticks=x, xticklabels=short_names,
            ylabel=r"$\Delta$ information criterion",
            title="Fit improvement after complexity penalty")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

delta_chi2 = comparison.loc[0, "chi2_min"] - comparison.loc[1, "chi2_min"]
print(f"Model 2 improves chi2 by {delta_chi2:.3f} while adding 2 parameters.")
print("AIC weakly prefers Model 2; BIC prefers the more economical Model 1.")


**What the criterion plot says.** Two extra parameters lower Model 2's recorded $\chi^2$ by $4.798$. AIC judges that gain just large enough to prefer Model 2, but only by $\Delta\mathrm{AIC}=0.798$. BIC's $k\ln N$ penalty instead prefers Model 1 by $\Delta\mathrm{BIC}=4.639$. The trade-off is explicit: the more flexible model fits better, but the preferred model changes with the strength and purpose of the complexity penalty. Neither difference is an absolute verdict.


## 6. Bayesian evidence and Bayes factors

For model $\mathcal M$, the evidence (marginal likelihood) is

$$Z_{\mathcal M}=p(D\mid\mathcal M)=\int \mathcal L(\theta)\,\pi(\theta\mid\mathcal M)\,d\theta,$$

and the Bayes factor comparing Models 1 and 2 is $B_{12}=Z_1/Z_2$. Evidence averages the likelihood over the **entire prior volume**, which supplies a Bayesian Occam penalty. It is therefore sensitive to prior ranges and parameterization.

- **AIC/BIC:** no chain is required if $\log\mathcal L_{\max}$ is known.
- **Contours and credible intervals:** use converged posterior samples.
- **Saved `emcee` samples plus `loglike`:** enough for posterior summaries and approximate evidence diagnostics, but not a direct numerical $Z$ because the posterior's unknown normalization is precisely the evidence.
- **Dedicated evidence methods:** nested sampling, thermodynamic integration, or sequential Monte Carlo estimate the normalization using additional sampling structure.

BIC has a large-sample evidence connection,

$$\log B_{12}\approx-\tfrac12(\mathrm{BIC}_1-\mathrm{BIC}_2),$$

but it is not an exact identity and does not expose explicit prior-width sensitivity.


In [ ]:
bic1, bic2 = comparison.loc[0, "BIC"], comparison.loc[1, "BIC"]
log_B12_bic = 0.5 * (bic2 - bic1)
B12_bic = np.exp(log_B12_bic)
p_model1_equal_prior_odds = B12_bic / (1.0 + B12_bic)

print("BIC large-sample approximation (not a direct evidence calculation):")
print(f"  log B_12 ≈ {log_B12_bic:.3f}")
print(f"  B_12 ≈ {B12_bic:.2f}:1 in favour of Model 1")
print("  With equal prior odds for only these two models, "
      f"P(Model 1 | D) ≈ {p_model1_equal_prior_odds:.3f}")


### What is the Laplace evidence approximation?

Let $\hat\theta$ be one posterior mode and

$$H=-\nabla^2\log[\mathcal L(\theta)\pi(\theta)]\big|_{\hat\theta}.$$

The Laplace method replaces the log posterior near that mode by its quadratic Taylor expansion, making the local integrand a multivariate Gaussian:

$$\log Z_{\rm Lap}\approx\log\mathcal L(\hat\theta)+\log\pi(\hat\theta)
+\frac{k}{2}\log(2\pi)-\frac12\log|H|.$$

For independent uniform priors with total volume $V_{\rm prior}$ and an interior mode, $\Sigma=H^{-1}$ gives

$$\log Z_{\rm Lap}\approx\log\mathcal L(\hat\theta)
+\frac{k}{2}\log(2\pi)+\frac12\log|\Sigma|-\log V_{\rm prior}.$$

In words: **peak likelihood × effective posterior volume / prior volume**. The volume ratio is the Occam factor.

Strict Laplace uses the inverse Hessian at the mode. Here we substitute the covariance of the saved posterior draws, so the calculation is more precisely a **covariance-Gaussian posterior approximation**. Curved or multimodal tails, strong degeneracies, prior truncation, or a mode near a boundary can make it unreliable.

There is also a provenance warning worth teaching: the local run configuration gives $M_2\in[-0.1,0.1]$, while Table 1 of the paper states $[-1,1]$. The export itself reaches the narrower boundaries. We therefore report the narrow-run calculation and show the paper-width result only as an algebraic sensitivity test; it is not a re-analysis under the wider prior.


In [ ]:
def covariance_gaussian_log_evidence(samples, loglike_at_peak, prior_bounds):
    'Gaussian-posterior evidence proxy for independent uniform priors.'
    samples = np.asarray(samples, dtype=float)
    bounds = np.asarray(prior_bounds, dtype=float)
    if samples.shape[1] != len(bounds):
        raise ValueError("One prior interval is required for each parameter.")
    widths = bounds[:, 1] - bounds[:, 0]
    if np.any(widths <= 0):
        raise ValueError("Every prior upper bound must exceed its lower bound.")
    covariance = np.cov(samples, rowvar=False, ddof=1)
    sign, logdet_covariance = np.linalg.slogdet(covariance)
    if sign <= 0:
        raise ValueError("Posterior covariance is not positive definite.")
    k = samples.shape[1]
    log_prior_volume = np.log(widths).sum()
    logZ = (loglike_at_peak + 0.5 * k * np.log(2 * np.pi)
            + 0.5 * logdet_covariance - log_prior_volume)
    correlation_condition = np.linalg.cond(np.corrcoef(samples, rowvar=False))
    return logZ, log_prior_volume, logdet_covariance, correlation_condition

# Common uniform ranges. Verify these against frozen run metadata before publication use.
bounds_model1 = [
    (8.0, 13.0), (0.01, 1.2), (-0.1, 0.1), (-0.1, 0.1),
    (0.01, 1.5), (0.001, 2.0), (0.001, 0.5),
]
bounds_model2_local_run = [
    (8.0, 13.0), (-0.1, 0.1), (-0.01, 0.01),
    (0.01, 1.2), (-0.1, 0.1), (-0.1, 0.1),
    (0.01, 1.5), (0.001, 2.0), (0.001, 0.5),
]
bounds_model2_paper_table = [
    (8.0, 13.0), (-1.0, 1.0), (-0.01, 0.01),
    (0.01, 1.2), (-0.1, 0.1), (-0.1, 0.1),
    (0.01, 1.5), (0.001, 2.0), (0.001, 0.5),
]

peak_loglike_model1 = comparison.loc[0, "loglike_max"]
peak_loglike_model2 = comparison.loc[1, "loglike_max"]
lap1 = covariance_gaussian_log_evidence(joint_chain, peak_loglike_model1, bounds_model1)
lap2_local = covariance_gaussian_log_evidence(
    model2_chain, peak_loglike_model2, bounds_model2_local_run
)
lap2_paper = covariance_gaussian_log_evidence(
    model2_chain, peak_loglike_model2, bounds_model2_paper_table
)

laplace_comparison = pd.DataFrame([
    ["Model 1", "common local/paper ranges", 7, peak_loglike_model1, *lap1],
    ["Model 2", "local run: M2 in [-0.1, 0.1]", 9, peak_loglike_model2, *lap2_local],
    ["Model 2", "paper-width sensitivity: M2 in [-1, 1]", 9, peak_loglike_model2, *lap2_paper],
], columns=["model", "prior_scenario", "k", "peak_loglike", "logZ_Gaussian",
            "log_prior_volume", "logdet_covariance", "corr_condition"])
display(laplace_comparison.round(3))

log_B12_local = lap1[0] - lap2_local[0]
log_B12_paper_width = lap1[0] - lap2_paper[0]
B21_local = np.exp(-log_B12_local)
B12_paper_width = np.exp(log_B12_paper_width)
print("Covariance-Gaussian / Laplace teaching approximation:")
print(f"  Local-run prior: log B_12 ≈ {log_B12_local:.3f}; "
      f"B_21 ≈ {B21_local:.2f}:1 (nominally Model 2)")
print(f"  Paper-width algebraic sensitivity: log B_12 ≈ {log_B12_paper_width:.3f}; "
      f"B_12 ≈ {B12_paper_width:.2f}:1 (nominally Model 1)")
print("  The sign flips when one prior width changes by a factor of ten: this is the lesson, not a robust model verdict.")

def edge_fraction_table(samples, bounds, names, fractional_width=0.01):
    bounds = np.asarray(bounds, dtype=float)
    widths = bounds[:, 1] - bounds[:, 0]
    near_lower = (samples - bounds[:, 0]) <= fractional_width * widths
    near_upper = (bounds[:, 1] - samples) <= fractional_width * widths
    return pd.DataFrame({
        "parameter": names,
        "sample_min": samples.min(axis=0), "sample_max": samples.max(axis=0),
        "percent_near_lower_1pct": 100 * near_lower.mean(axis=0),
        "percent_near_upper_1pct": 100 * near_upper.mean(axis=0),
    })

print("Model-2 boundary diagnostic under the local-run prior:")
display(edge_fraction_table(
    model2_chain, bounds_model2_local_run, model2_names
).round(4))
print("Absolute logZ values inherit any model-independent likelihood constant omitted from the saved loglike.")
print("It cancels in B12 only because both models use the same data, errors, and omitted constant.")


**How to read the Laplace result.** BIC's asymptotic approximation favours Model 1, while the covariance-Gaussian calculation with the local narrow $M_2$ prior nominally favours Model 2. Merely widening that one prior by a factor of ten reverses the Gaussian result. In addition, the Model-2 samples approach several prior edges and the correlation matrices are ill-conditioned. The calculation is therefore useful as a short-course demonstration of posterior volume and prior sensitivity, but it is not a publication-quality Bayes factor.


### Direct numerical evidence template with nested sampling

Nested sampling does not require a pre-existing `emcee` chain. It creates its own live points, estimates $\log Z$ with a numerical error, and also produces weighted posterior samples. “Direct numerical” is more accurate language than “exact”: the result still has Monte Carlo error and model/likelihood systematics.

The full $N=112$ data and reusable local likelihood are now connected below, but the run is disabled because it is computationally expensive and the HMF/prior provenance issues above should be resolved first. Select the intended Model-2 prior explicitly, align the HMF calibration with the paper, and run more than once to check numerical stability. The normalized Gaussian likelihood is used so that absolute $\log Z$ has a defined additive constant.


In [ ]:
def make_uniform_prior_transform(bounds):
    'Map a unit-cube point to independent uniform parameter priors.'
    bounds = np.asarray(bounds, dtype=float)
    lower, upper = bounds[:, 0], bounds[:, 1]
    def prior_transform(unit_cube):
        return lower + np.asarray(unit_cube) * (upper - lower)
    return prior_transform

def parameters_from_theta(theta, model):
    theta = np.asarray(theta, dtype=float)
    if model == 1:
        m1, alpha0, alpha1, alpha2, beta0, sigma0, epsilon0 = theta
        return dict(logM0=m1, alpha0=alpha0, alpha1=alpha1, alpha2=alpha2,
                    beta=beta0, sigma_uv=sigma0, epsilon0=epsilon0)
    if model == 2:
        m1, m2, m3, alpha0, alpha1, alpha2, beta0, sigma0, epsilon0 = theta
        return dict(M1=m1, M2=m2, M3=m3, alpha0=alpha0, alpha1=alpha1,
                    alpha2=alpha2, beta=beta0, sigma_uv=sigma0, epsilon0=epsilon0)
    raise ValueError("model must be 1 or 2")

def normalized_joint_loglike(theta, model):
    parameters = parameters_from_theta(theta, model)
    total = 0.0
    for z in JOINT_REDSHIFTS:
        alpha, logM0 = physical_parameters_at_z(parameters, z)
        if alpha <= 0 or not (8 <= logM0 <= 13) or parameters["sigma_uv"] <= 0:
            return -np.inf
        muv, phi_model = physical_uvlf_curve(z, parameters)
        observations = joint_data_by_z[z]
        order = np.argsort(muv)
        if (observations.Muv.min() < muv[order].min()
                or observations.Muv.max() > muv[order].max()):
            return -np.inf
        prediction = np.interp(observations.Muv, muv[order], phi_model[order])
        variance = observations.sigma_eff.to_numpy() ** 2
        residual = observations.phi.to_numpy() - prediction
        total += -0.5 * np.sum(residual**2 / variance + np.log(2 * np.pi * variance))
    return total

def loglike_model1(theta):
    return normalized_joint_loglike(theta, model=1)

def loglike_model2(theta):
    return normalized_joint_loglike(theta, model=2)

def run_nested_evidence(loglike, bounds, nlive=500, dlogz=0.1):
    'Return log evidence, its numerical error, and the full dynesty result.'
    try:
        from dynesty import NestedSampler
    except ImportError as exc:
        raise ImportError("Install dynesty in this notebook kernel before this run.") from exc
    sampler = NestedSampler(
        loglike, make_uniform_prior_transform(bounds), ndim=len(bounds),
        nlive=nlive, bound="multi", sample="rwalk",
    )
    sampler.run_nested(dlogz=dlogz, print_progress=True)
    result = sampler.results
    return result.logz[-1], result.logzerr[-1], result

# Nested sampling does not use the saved posterior chain, so choose the intended prior directly.
MODEL2_NESTED_PRIOR = "paper_table"  # change to "local_run" only if that is the target model
nested_bounds_model2 = (
    bounds_model2_paper_table if MODEL2_NESTED_PRIOR == "paper_table"
    else bounds_model2_local_run
)
RUN_NUMERICAL_EVIDENCE = False
if RUN_NUMERICAL_EVIDENCE:
    logZ1, logZerr1, nested1 = run_nested_evidence(loglike_model1, bounds_model1)
    logZ2, logZerr2, nested2 = run_nested_evidence(loglike_model2, nested_bounds_model2)
    print(f"log Z1 = {logZ1:.3f} ± {logZerr1:.3f}")
    print(f"log Z2 = {logZ2:.3f} ± {logZerr2:.3f}")
    print(f"log B12 = {logZ1 - logZ2:.3f}")
else:
    print("Nested evidence skipped. Resolve the HMF/prior provenance, install dynesty, then set RUN_NUMERICAL_EVIDENCE=True.")


## 7. Interpretation and short-course exercises

Model 2 lowers the recorded $\chi^2$ by about $4.80$ using two additional parameters. AIC gives that improvement slightly more credit and weakly favours Model 2; BIC's stronger penalty at $N=112$ favours Model 1 by $\Delta\mathrm{BIC}\simeq4.64$. The BIC approximation corresponds to $B_{12}\simeq10.2$, but the covariance-Gaussian evidence proxy is both non-Gaussian/boundary-sensitive and prior-sensitive enough to reverse its preference. “Best model” therefore depends on the target—prediction, economical description, or prior-averaged Bayesian support—and on whether the evidence assumptions are actually satisfied.

Suggested exercises:

1. Change `plot_stride` and verify that the GetDist contours remain stable. Explain why this is plot decimation rather than statistically necessary thinning.
2. Compare the $z=11$ best-fit and posterior-median curves and explain why they need not coincide.
3. Replace the symmetric effective error with a split-normal likelihood in the warm-up.
4. Inspect the per-redshift $\chi^2$ table. Identify where Model 2 gains and loses fit quality before looking at the total.
5. Change only the Model-2 $M_2$ prior width and predict the corresponding change in the Gaussian $\log Z$ before rerunning the cell.
6. Estimate a numerical Hessian at the mode and compare $H^{-1}$ with the chain covariance. Which contours make the comparison fail?
7. Align the HMF utility with the paper's stated calibration, verify the ten-bin likelihood against the recorded maxima, and then run nested sampling with at least two scientifically motivated prior choices.
8. Retain unflattened chains in the next MCMC run and add burn-in, autocorrelation-time, trace-plot, effective-sample-size, and between-chain convergence diagnostics.

### Take-home distinction

- **Fit quality:** $\chi^2_{\min}$ or $\log\mathcal L_{\max}$.
- **Predictive complexity trade-off:** AIC/AICc.
- **Sample-size-penalized approximation:** BIC.
- **Bayesian model probability update:** evidence/Bayes factor, conditional on explicit priors and a normalized likelihood.
- **Parameter uncertainty and degeneracy:** converged posterior samples and plots such as GetDist.
